In [0]:
%sql
select current_catalog();

In [0]:
%sql
show CATALOGS;

In [0]:
spark.sql("use CATALOG `databricks-pyspark` ")

In [0]:
spark.sql("use SCHEMA `databricks-pyspark-schema` ")

In [0]:
%sql 
SHOW TABLES;

In [0]:
data = [
    (1,"John","NewYork")
]
from pyspark.sql.functions import lit, to_date
initial_df = spark.createDataFrame(data, ["id", "name", "address"]) \
    .withColumn('from_date',to_date(lit('2025-05-05'))) \
        .withColumn('end_date',lit(None).cast('date')).withColumn('is_current',lit('Y'))


In [0]:
initial_df.write.mode("overwrite").saveAsTable('initial_date')

In [0]:
data = [
    (1,"John","Chicago")
]
changed_df = spark.createDataFrame(data,['id','name','address']) \
                        .withColumn('changed_date',to_date(lit('2025-05-06')))
                        
                            


In [0]:
changed_df.write.mode("overwrite").format("delta").saveAsTable("emp_changes_log")

In [0]:
spark.sql("select * from emp_changes_log").show()

In [0]:
spark.sql("""
          MERGE INTO initial_date as t 
          USING emp_changes_log s ON s.id = t.id
                AND t.address <> s.address
                AND t.is_current = 'Y'
          WHEN MATCHED THEN UPDATE SET t.is_current = 'N',
                            t.end_date = s.changed_date
          """)

In [0]:
%sql
select * from initial_date;

In [0]:
final_df = spark.sql("""
                     select s.id,
                     s.name,
                     s.address,
                     s.changed_date as effective_date,
                     cast(null as date) as end_date,
                     'Y' as is_current 
                      from emp_changes_log s LEFT join initial_date t on s.id = t.id AND t.is_current = 'Y'
                      where t.id is null
                     """)

final_df.show()                     

In [0]:
final_df.write.mode("overwrite").saveAsTable("final_date")